# Scrap Serenity (@aleabitoreddit)

X(Twitter) 트윗 수집 → `analysis_Serenity.db`

- **post**: 일반 게시물
- **reply**: 답글
- **subscriber**: 구독자 전용 (`exclusivityInfo` 기반 감지)

## 1. Configuration

쿠키는 `.env` 파일에서 로드됩니다. 만료 시 `.env`의 `X_AUTH_TOKEN`, `X_CT0`, `X_TWID` 값을 교체하세요.

In [1]:
# ── Cookie ────────────────────────────────────────────────────────────────────
# X(Twitter) 쿠키 — 만료 시 .env 파일에서 새 값으로 교체
import os
from dotenv import load_dotenv

_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
load_dotenv(os.path.join(_notebook_dir, '.env'))

COOKIES = {
    "auth_token": os.environ["X_AUTH_TOKEN"],
    "ct0": os.environ["X_CT0"],
    "twid": os.environ["X_TWID"],
}

# ── Target ────────────────────────────────────────────────────────────────────
TARGET_USER = "aleabitoreddit"

# ── Options ───────────────────────────────────────────────────────────────────
TWEET_COUNT = None   # None = 전체, 숫자 = 최대 N개
DEBUG = False        # True = raw tweet._data JSON 덤프

# ── Paths ─────────────────────────────────────────────────────────────────────
DB_DIR = os.path.join(_notebook_dir, 'References')
DB_PATH = os.path.join(DB_DIR, 'analysis_Serenity.db')

os.makedirs(DB_DIR, exist_ok=True)
print(f"DB path: {DB_PATH}")

DB path: /Users/seongjin/Documents/⭐성진이의 옵시디언/💶Invest/.claude/skills/Serenity/References/analysis_Serenity.db


## 2. Setup

In [2]:
import asyncio
import sqlite3
import json
import re
import tempfile
from datetime import datetime, timedelta, timezone
from twikit import Client

# twikit 2.3.3 버그 우회: User.__init__이 legacy의 여러 키를 .get() 없이 직접 접근해 KeyError.
# X API 응답에서 누락 가능한 키들을 미리 기본값으로 채워 넣는다.
import twikit.user as _twikit_user
_orig_user_init = _twikit_user.User.__init__
_LEGACY_DEFAULTS = {
    'created_at': '', 'name': '', 'screen_name': '', 'profile_image_url_https': '',
    'location': '', 'description': '', 'pinned_tweet_ids_str': [],
    'verified': False, 'possibly_sensitive': False, 'can_dm': False, 'can_media_tag': False,
    'want_retweets': False, 'default_profile': False, 'default_profile_image': False,
    'has_custom_timelines': False, 'is_translator': False, 'translator_type': 'none',
    'followers_count': 0, 'fast_followers_count': 0, 'normal_followers_count': 0,
    'friends_count': 0, 'favourites_count': 0, 'listed_count': 0,
    'media_count': 0, 'statuses_count': 0, 'withheld_in_countries': [],
}
def _patched_user_init(self, client, data):
    data.setdefault('is_blue_verified', False)
    legacy = data.setdefault('legacy', {})
    for _k, _v in _LEGACY_DEFAULTS.items():
        legacy.setdefault(_k, _v)
    legacy.setdefault('entities', {}).setdefault('description', {}).setdefault('urls', [])
    _orig_user_init(self, client, data)
_twikit_user.User.__init__ = _patched_user_init

KST = timezone(timedelta(hours=9))
RATE_LIMIT_DELAY = 2
DUPLICATE_THRESHOLD = 10
MAX_RETRIES = 3


def to_kst(twitter_time_str):
    dt = datetime.strptime(twitter_time_str, "%a %b %d %H:%M:%S %z %Y")
    return dt.astimezone(KST).strftime("%Y-%m-%dT%H:%M:%S+09:00")


def extract_tickers(text):
    tickers = re.findall(r'\$([A-Za-z]+)', text)
    return list(dict.fromkeys([t.upper() for t in tickers]))


def get_full_content(tweet):
    note = (tweet._data
            .get('note_tweet', {})
            .get('note_tweet_results', {})
            .get('result', {}))
    if note and 'text' in note:
        return note['text']
    return tweet.full_text


def get_media_urls(tweet):
    urls = []
    if tweet.media:
        for m in tweet.media:
            if hasattr(m, 'media_url') and m.media_url:
                urls.append(m.media_url)
    return urls


def classify_tweet_type(tweet):
    if tweet._data.get('exclusivityInfo'):
        return 'subscriber'
    elif tweet.in_reply_to:
        return 'reply'
    else:
        return 'post'


def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS tweets (
            id TEXT PRIMARY KEY,
            user TEXT NOT NULL,
            type TEXT NOT NULL CHECK(type IN ('post', 'reply', 'subscriber')),
            created_at TEXT NOT NULL,
            content TEXT,
            tickers TEXT DEFAULT '[]',
            media TEXT DEFAULT '[]'
        )
    ''')
    conn.commit()
    return conn


def get_existing_ids(conn):
    cur = conn.execute('SELECT id FROM tweets')
    return set(row[0] for row in cur.fetchall())


def save_tweets(conn, tweets):
    saved = 0
    for tweet in tweets:
        content = get_full_content(tweet)
        tickers = json.dumps(extract_tickers(content), ensure_ascii=False)
        media = json.dumps(get_media_urls(tweet), ensure_ascii=False)
        tweet_type = classify_tweet_type(tweet)
        conn.execute(
            'INSERT OR IGNORE INTO tweets (id, user, type, created_at, content, tickers, media) VALUES (?,?,?,?,?,?,?)',
            (tweet.id, tweet.user.screen_name if tweet.user else TARGET_USER,
             tweet_type, to_kst(tweet.created_at), content, tickers, media))
        if conn.total_changes:
            saved += 1
    conn.commit()
    return saved


async def fetch_with_retry(coro_fn):
    for attempt in range(MAX_RETRIES):
        try:
            return await coro_fn()
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                raise
            wait = 2 ** (attempt + 1)
            print(f"  ⚠ {e} — retry in {wait}s...")
            await asyncio.sleep(wait)


print("Setup complete.")

Setup complete.


## 3. Scrape

In [3]:
async def run_scrape():
    conn = init_db()
    existing_ids = get_existing_ids(conn)
    print(f"DB: {DB_PATH}")
    print(f"Existing: {len(existing_ids)}")

    # 쿠키를 임시 파일로 저장하여 twikit에 전달
    cookie_file = tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False)
    json.dump(COOKIES, cookie_file)
    cookie_file.close()

    client = Client('en-US')
    try:
        client.load_cookies(cookie_file.name)
    except Exception as e:
        print(f"✗ Cookie expired — update .env (X_AUTH_TOKEN, X_CT0, X_TWID).\n  {e}")
        conn.close()
        return
    finally:
        os.unlink(cookie_file.name)

    user = await client.get_user_by_screen_name(TARGET_USER)
    print(f"\n=== {user.name} (@{user.screen_name}) ===")
    print(f"Followers: {user.followers_count:,} | Tweets: {user.statuses_count:,}")

    total_saved = 0
    type_counts = {'post': 0, 'reply': 0, 'subscriber': 0}

    for tweet_type in ['Tweets', 'Replies']:
        print(f"\nFetching {tweet_type}...")
        consecutive_dups = 0
        total_fetched = 0

        try:
            tweets = await fetch_with_retry(
                lambda tt=tweet_type: user.get_tweets(tt, count=20))
        except Exception as e:
            print(f"  ⚠ Failed: {e}")
            continue

        while tweets:
            page_tweets = []
            for t in tweets:
                sn = t.user.screen_name if t.user else None
                if sn and sn.lower() != TARGET_USER.lower():
                    continue

                total_fetched += 1

                if DEBUG:
                    print(f"  [DEBUG] {t.id}: exclusivityInfo={t._data.get('exclusivityInfo')}, in_reply_to={t.in_reply_to}")

                if t.id in existing_ids:
                    consecutive_dups += 1
                    if consecutive_dups >= DUPLICATE_THRESHOLD:
                        print(f"  {DUPLICATE_THRESHOLD} consecutive dups — stopping")
                        break
                else:
                    consecutive_dups = 0
                    tt = classify_tweet_type(t)
                    type_counts[tt] += 1
                    page_tweets.append(t)
                    existing_ids.add(t.id)
                    if TWEET_COUNT and (total_saved + len(page_tweets)) >= TWEET_COUNT:
                        break

            if page_tweets:
                saved = save_tweets(conn, page_tweets)
                total_saved += saved
                print(f"  Fetched {total_fetched}, saved {total_saved} total")

            if consecutive_dups >= DUPLICATE_THRESHOLD:
                break
            if TWEET_COUNT and total_saved >= TWEET_COUNT:
                break

            await asyncio.sleep(RATE_LIMIT_DELAY)
            try:
                tweets = await fetch_with_retry(lambda: tweets.next())
                if not tweets:
                    break
            except Exception as e:
                print(f"  ⚠ Pagination stopped: {e}")
                break

        if TWEET_COUNT and total_saved >= TWEET_COUNT:
            break

    print(f"\n{'='*40}")
    print(f"Saved this run: {total_saved}")
    print(f"  post={type_counts['post']}, reply={type_counts['reply']}, subscriber={type_counts['subscriber']}")

    # DB 통계
    cur = conn.execute('SELECT type, COUNT(*) FROM tweets GROUP BY type')
    stats = cur.fetchall()
    total = sum(c for _, c in stats)
    print(f"\nTotal in DB: {total}")
    for t, c in stats:
        print(f"  {t}: {c}")
    conn.close()

await run_scrape()

DB: /Users/seongjin/Documents/⭐성진이의 옵시디언/💶Invest/.claude/skills/Serenity/References/analysis_Serenity.db
Existing: 1053

=== Serenity (@aleabitoreddit) ===
Followers: 232,967 | Tweets: 5,961

Fetching Tweets...
  10 consecutive dups — stopping
  Fetched 17, saved 7 total

Fetching Replies...
  ⚠ status: 404, message: "" — retry in 2s...
  ⚠ status: 404, message: "" — retry in 4s...
  ⚠ Failed: status: 404, message: ""

Saved this run: 7
  post=7, reply=0, subscriber=0

Total in DB: 1060
  post: 921
  reply: 1
  subscriber: 138


## 4. Explore DB

In [4]:
# ── 최신 트윗 확인 ─────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    'SELECT id, type, created_at, substr(content,1,80), tickers FROM tweets ORDER BY created_at DESC LIMIT 10')
for row in cur:
    print(f"[{row[1]:10s}] {row[2]}  {row[3]}")
    if row[4] != '[]':
        print(f"             tickers: {row[4]}")
conn.close()

[post      ] 2026-04-26T16:20:39+09:00  Agreed, and glad DNB, one of Europe's leading banks, went out to defend $SIVE va
             tickers: ["SIVE", "AAPL", "JBL", "MRVL", "POET", "AMD", "NVDA", "AMZN", "MSFT", "LITE"]
[post      ] 2026-04-26T14:18:19+09:00  I swear I had 50,000 followers and 0 subscribers like 2 months ago.

And now I r
[post      ] 2026-04-26T06:42:54+09:00  I wasn't joking when I said $IREN community members have low IQ. 

Since they li
             tickers: ["IREN"]
[post      ] 2026-04-26T05:52:56+09:00  I have high conviction that majority/full port $IREN investors are the dumbest p
             tickers: ["IREN"]
[post      ] 2026-04-26T05:00:04+09:00  It's highly nuanced, and I'll explain why it's not late, but late to some: 

Pho
             tickers: ["LITE", "COHR", "AXTI", "AAOI", "JBL", "SIVE", "POET", "ALMU"]
[post      ] 2026-04-26T04:34:52+09:00  Uh chat... is my timing insane or what?

Names like $TSEM are flat all year, I g
             tickers: ["T

In [5]:
# ── 타입별 통계 ───────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute('SELECT type, COUNT(*) FROM tweets GROUP BY type')
for t, c in cur:
    print(f"  {t}: {c}")
total = conn.execute('SELECT COUNT(*) FROM tweets').fetchone()[0]
print(f"  total: {total}")
conn.close()

  post: 921
  reply: 1
  subscriber: 138
  total: 1060


In [6]:
# ── 구독자 전용 트윗 확인 ─────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    "SELECT created_at, substr(content,1,100), tickers FROM tweets WHERE type='subscriber' ORDER BY created_at DESC LIMIT 10")
for row in cur:
    print(f"{row[0]}  {row[1]}")
    if row[2] != '[]':
        print(f"  tickers: {row[2]}")
    print()
conn.close()

2026-04-22T19:47:04+09:00  My neighbors just keep going at it every day with a lot of noise. It's super annoying. 

That aside,
  tickers: ["DOW"]

2026-04-22T04:46:40+09:00  There’s always going to be new opportunities in the markets.

like scale across with $MXL tam expans
  tickers: ["MXL", "EWY", "COHR"]

2026-04-20T18:31:55+09:00  Two random stocks taking a look into... 

Skytech (6937) - $722m, CMIt (7853 TWSE?) 

Skytech - PvD/
  tickers: ["KLAC"]

2026-04-19T04:03:31+09:00  MLCC, inductor + PCB drill bottleneck: 

- Murata (TSE: 6981) 
- Taiyo Yuden(TSE: 6976), third large
  tickers: ["VSH"]

2026-04-16T06:31:37+09:00  So my opinion on current markets and thoughts:

Lot of photonics stuff ran up too much like $LITE, $
  tickers: ["LITE", "AXTI", "AAOI", "AEHR", "SMTC", "META", "MSFT", "RDDT", "CRCL", "MRVL", "NBIS", "GOOGL", "INTC", "ARM", "EWY", "HOOD", "RKLB", "TSEM"]

2026-04-15T13:30:09+09:00  okay so few i was interested in:

PCL - broadcom optical foundry $494m MC
fittech

In [7]:
# ── 특정 티커 검색 ────────────────────────────────────────────────────────
SEARCH_TICKER = "AXTI"  # ← 변경하여 검색

conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    "SELECT type, created_at, substr(content,1,100) FROM tweets WHERE tickers LIKE ? ORDER BY created_at DESC",
    (f'%"{SEARCH_TICKER}"%',))
rows = cur.fetchall()
print(f"${SEARCH_TICKER} mentions: {len(rows)}")
for row in rows[:10]:
    print(f"  [{row[0]}] {row[1]}  {row[2]}")
conn.close()

$AXTI mentions: 157
  [post] 2026-04-26T05:00:04+09:00  It's highly nuanced, and I'll explain why it's not late, but late to some: 

Photonics is the newest
  [post] 2026-04-26T01:54:25+09:00  Two most viral stories on $RDDT:

1. Turning $252K -> $7.7M with $AMD
2. Turning $167K -> $2.2M with
  [post] 2026-04-25T06:30:44+09:00  Did you listen anon?

The fact that $SIVE is up 600%+.

But still can 10x from here in a year... onc
  [post] 2026-04-23T09:29:10+09:00  There's a reason I spotlight EU small caps.

This my investment thesis that I haven't publicly state
  [post] 2026-04-22T20:00:20+09:00  Bro everyone was doubting me on names like $IQE.

Yet how come… all these institutions from UBS or P
  [post] 2026-04-22T09:34:42+09:00  I’m genuinely impressed one of you copy traded your way into managing a hedge fund.

But please don’
  [post] 2026-04-20T08:11:51+09:00  Frontrunning 1.6T/CPO within the broader photonics supercycle is the most compelling investment to m
  [post] 2026-04-20T0